# 05 – Conformal prediction : des intervalles de RUL garantis

**Objectif :** au lieu de dire « il reste 60 cycles », dire « avec 90 % de garantie, il reste entre 45 et 80 cycles », ou « avec 95 % de garantie, il reste **au moins** 45 cycles ».

**Plan :**
1. Construire un ensemble de calibration qui ressemble au jeu de test
2. Split conformal codé à la main, puis vérifié avec MAPIE
3. Vérifier la couverture : globalement, puis par tranche de RUL
4. Intervalles adaptatifs : CQR (MAPIE) et Mondrian (fait main)
5. Borne basse garantie, la sortie utile pour la maintenance

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from mapie.regression import SplitConformalRegressor, ConformalizedQuantileRegressor

from src.data.load import PROJECT_ROOT
from src.features.preprocess import FEATURES_FD001
from src.features.rolling import add_rolling_features, last_cycle
from src.conformal.conformal import (sample_cuts, conformal_quantile, SplitConformal,
                                     MondrianConformal, LowerBoundConformal, coverage, mean_width)

sns.set_theme(style="whitegrid")
PROC = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"
ALPHA = 0.10   # niveau de confiance visé : 90 %
BINS = [-1, 25, 50, 75, 100, 125]
LABELS = ["0-25", "25-50", "50-75", "75-100", "100-125"]

## 1. Données et modèle LightGBM de l'étape 3

In [ ]:
sets = {k: add_rolling_features(pd.read_csv(PROC / f"FD001_{k}.csv"), FEATURES_FD001, windows=(5, 10, 20))
        for k in ["train", "calib", "val", "test"]}
bundle = joblib.load(MODELS / "lgbm_fd001.joblib")
lgbm, FEATS = bundle["model"], bundle["features"]

## 2. Un ensemble de calibration qui ressemble au test

La garantie de la conformal prediction suppose que les données de calibration et les nouvelles données sont **échangeables** (elles se comportent statistiquement de la même façon).

Or on a vu à l'étape 3 que le dernier cycle d'une trajectoire complète (RUL = 0) n'a rien à voir avec le dernier cycle d'une trajectoire de test coupée. On **coupe donc les moteurs de calibration à des cycles aléatoires** (au moins 31 cycles observés, comme dans le test). On prend 20 coupures par moteur pour avoir assez de points : 15 moteurs × 20 = 300 points.

Même chose pour la validation, qui servira à vérifier les intervalles sur des moteurs jamais vus.

In [ ]:
cal = sample_cuts(sets["calib"], n_cuts=20, seed=0)
val_cuts = sample_cuts(sets["val"], n_cuts=20, seed=1)
test_last = last_cycle(sets["test"])

for name, d in [("calibration", cal), ("validation (coupée)", val_cuts), ("test (dernier cycle)", test_last)]:
    d["pred"] = np.clip(lgbm.predict(d[FEATS]), 0, None)
    print(f"{name:<22}: {len(d):>4} points, {d.unit.nunique():>3} moteurs")

fig, ax = plt.subplots(figsize=(8, 4))
for name, d in [("calibration (coupée)", cal), ("test (dernier cycle)", test_last)]:
    ax.hist(d.RUL, bins=np.arange(0, 131, 10), alpha=0.5, density=True, label=name)
ax.set(xlabel="Vraie RUL (plafonnée)", ylabel="Densité", title="La calibration ressemble-t-elle au test ?")
ax.legend()
plt.show()

## 3. Split conformal codé à la main

1. Scores de non-conformité sur la calibration : s_i = |y_i − ŷ_i|
2. q = le k-ième plus petit score, avec k = ⌈(n+1)(1−α)⌉
3. Intervalle : [ŷ − q, ŷ + q]

Le « +1 » est la correction d'échantillon fini : c'est elle qui rend la garantie exacte, et pas seulement asymptotique.

In [ ]:
scores = np.abs(cal.RUL - cal.pred)
n = len(scores)
k = int(np.ceil((n + 1) * (1 - ALPHA)))
q = conformal_quantile(scores, ALPHA)
print(f"n = {n} scores, k = ⌈{n + 1} × {1 - ALPHA}⌉ = {k}  →  q = {q:.2f} cycles")
print(f"Intervalle à 90 % : prédiction ± {q:.1f} cycles")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores, bins=40, color="steelblue")
ax.axvline(q, color="red", ls="--", label=f"q = {q:.1f}")
ax.set(xlabel="|vraie RUL − prédiction|", ylabel="Nombre", title="Scores de non-conformité sur la calibration")
ax.legend()
plt.show()

### Vérification avec MAPIE

MAPIE doit donner exactement le même intervalle. `prefit=True` : on lui donne le modèle déjà entraîné, il ne fait que la calibration.

In [ ]:
mapie_split = SplitConformalRegressor(estimator=lgbm, confidence_level=1 - ALPHA, prefit=True)
mapie_split.conformalize(cal[FEATS], cal.RUL)
_, itv = mapie_split.predict_interval(test_last[FEATS])
half_width_mapie = (itv[:, 1, 0] - itv[:, 0, 0]).mean() / 2
print(f"Demi-largeur à la main : {q:.3f}  |  MAPIE : {half_width_mapie:.3f}")

## 4. La couverture est-elle tenue ?

La garantie dit : au moins 90 % des vraies RUL doivent tomber dans l'intervalle. On vérifie sur des moteurs **jamais vus**, pour plusieurs niveaux de confiance.

In [ ]:
rows = []
for level in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    sc = SplitConformal(alpha=1 - level).fit(cal.pred, cal.RUL)
    for name, d in [("validation (coupée)", val_cuts), ("test (dernier cycle)", test_last)]:
        lo, hi = sc.predict(d.pred)
        rows.append({"niveau visé": level, "ensemble": name, "couverture": coverage(d.RUL, lo, hi), "largeur": mean_width(lo, hi)})
cov = pd.DataFrame(rows)
print(cov.pivot(index="niveau visé", columns="ensemble", values=["couverture", "largeur"]).round(3))

fig, ax = plt.subplots(figsize=(6, 6))
for name, d in cov.groupby("ensemble"):
    ax.plot(d["niveau visé"], d["couverture"], "o-", label=name)
ax.plot([0.45, 1], [0.45, 1], "k--", label="couverture parfaite")
ax.set(xlabel="Niveau de confiance visé", ylabel="Couverture observée", title="Couverture empirique du split conformal")
ax.legend()
plt.show()

## 5. Le problème : la couverture par tranche de RUL

La garantie est **marginale** : 90 % en moyenne sur tous les moteurs. Mais on a vu à l'étape 3 que l'erreur dépend fortement de la RUL (faible près de la panne, forte en milieu de vie). Avec une largeur constante, on s'attend à des intervalles trop larges près de la panne et trop étroits au milieu.

In [ ]:
def by_bin(d, lo, hi, method):
    t = pd.DataFrame({"RUL": d.RUL.values, "lo": lo, "hi": hi})
    t["tranche"] = pd.cut(t.RUL, bins=BINS, labels=LABELS)
    t["couvert"] = (t.RUL >= t.lo) & (t.RUL <= t.hi)
    t["largeur"] = t.hi - t.lo
    out = t.groupby("tranche", observed=True).agg(couverture=("couvert", "mean"), largeur=("largeur", "mean"), n=("RUL", "size"))
    out["méthode"] = method
    return out.reset_index()

split = SplitConformal(alpha=ALPHA).fit(cal.pred, cal.RUL)
results = {}  # méthode -> (lo, hi) sur la validation coupée et le test
results["Split conformal"] = {name: split.predict(d.pred) for name, d in [("val", val_cuts), ("test", test_last)]}
by_bin(val_cuts, *results["Split conformal"]["val"], "Split conformal").round(3)

## 6. Intervalles adaptatifs n°1 : Conformalized Quantile Regression (CQR)

Au lieu d'un seul modèle de la moyenne, on entraîne LightGBM à prédire les **quantiles 5 % et 95 %** (perte « pinball »). (Le paramètre `alpha=0.5` est obligatoire pour MAPIE, qui le remplace ensuite par 0,05 / 0,5 / 0,95.) Ces quantiles s'écartent naturellement là où le modèle est incertain. Puis on les **conformalise** : sur la calibration, on mesure de combien il faut les élargir (ou les resserrer) pour garantir 90 %.

Score CQR : s_i = max(q̂_bas(x_i) − y_i, y_i − q̂_haut(x_i)), intervalle final [q̂_bas − q, q̂_haut + q].

MAPIE entraîne trois modèles LightGBM (quantiles 5 %, 50 %, 95 %) : compter 1 à 3 minutes.

In [ ]:
quantile_lgbm = lgb.LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=400, learning_rate=0.03, num_leaves=31,
                                  min_child_samples=50, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.7, random_state=42, verbose=-1)
cqr = ConformalizedQuantileRegressor(estimator=quantile_lgbm, confidence_level=1 - ALPHA)
cqr.fit(sets["train"][FEATS], sets["train"].RUL)
cqr.conformalize(cal[FEATS], cal.RUL)

results["CQR"] = {}
for name, d in [("val", val_cuts), ("test", test_last)]:
    _, itv = cqr.predict_interval(d[FEATS])
    results["CQR"][name] = (np.clip(itv[:, 0, 0], 0, None), itv[:, 1, 0])
print("CQR prêt")

## 7. Intervalles adaptatifs n°2 : Mondrian (fait main)

Idée plus simple : un quantile conformal **différent par tranche de RUL prédite**. On ne peut pas grouper par vraie RUL (inconnue au moment de prédire), mais on peut grouper par prédiction. La garantie devient : 90 % **dans chaque groupe**, et plus seulement en moyenne.

Limite : il faut assez de points de calibration dans chaque groupe.

In [ ]:
mond = MondrianConformal(alpha=ALPHA, bins=(0, 25, 50, 75, 100, np.inf)).fit(cal.pred, cal.RUL)
print(pd.DataFrame({"tranche prédite": ["0-25", "25-50", "50-75", "75-100", "100+"],
                    "points de calibration": mond.n_, "demi-largeur q": mond.q_.round(1)}))
results["Mondrian"] = {name: mond.predict(d.pred) for name, d in [("val", val_cuts), ("test", test_last)]}

## 8. Comparaison des trois méthodes

In [ ]:
summary, per_bin = [], []
for m, r in results.items():
    for name, d in [("val", val_cuts), ("test", test_last)]:
        lo, hi = r[name]
        bb = by_bin(d, lo, hi, m)
        summary.append({"méthode": m, "ensemble": name, "couverture": coverage(d.RUL, lo, hi),
                        "largeur moyenne": mean_width(lo, hi), "pire tranche": bb.couverture.min()})
        if name == "val":
            per_bin.append(bb)
summary = pd.DataFrame(summary)
per_bin = pd.concat(per_bin)
print("Objectif : couverture ≥ 0.90, avec la largeur la plus petite possible")
summary.pivot(index="méthode", columns="ensemble", values=["couverture", "largeur moyenne", "pire tranche"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=per_bin, x="tranche", y="couverture", hue="méthode", ax=axes[0])
axes[0].axhline(1 - ALPHA, color="red", ls="--", label="objectif 90 %")
axes[0].set(ylim=(0.5, 1.02), title="Couverture par tranche de vraie RUL (validation)", xlabel="Vraie RUL")
sns.barplot(data=per_bin, x="tranche", y="largeur", hue="méthode", ax=axes[1])
axes[1].set(title="Largeur moyenne des intervalles", xlabel="Vraie RUL", ylabel="Largeur (cycles)")
plt.tight_layout()
plt.show()

## 9. Les intervalles au fil de la vie des moteurs

Pour 4 moteurs de validation, on calcule l'intervalle à chaque cycle (à partir du cycle 31).

In [ ]:
full_val = sets["val"][sets["val"].cycle >= 31].copy()
full_val["pred"] = np.clip(lgbm.predict(full_val[FEATS]), 0, None)
full_val["split_lo"], full_val["split_hi"] = split.predict(full_val.pred)
_, itv = cqr.predict_interval(full_val[FEATS])
full_val["cqr_lo"], full_val["cqr_hi"] = np.clip(itv[:, 0, 0], 0, None), itv[:, 1, 0]

units = full_val.unit.unique()[:4]
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for j, u in enumerate(units):
    d = full_val[full_val.unit == u]
    for i, (m, lo, hi, c) in enumerate([("Split conformal", "split_lo", "split_hi", "steelblue"), ("CQR", "cqr_lo", "cqr_hi", "seagreen")]):
        ax = axes[i, j]
        ax.fill_between(d.cycle, d[lo], d[hi], color=c, alpha=0.25, label="intervalle 90 %")
        ax.plot(d.cycle, d.pred, color=c, lw=1, label="prédiction")
        ax.plot(d.cycle, d.RUL, "k--", lw=1, label="vraie RUL")
        ax.set(title=f"{m} – moteur {u}", xlabel="Cycle")
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 10. La sortie utile pour la maintenance : une borne basse garantie

Un ingénieur maintenance n'a pas besoin de la borne haute. Ce qu'il veut savoir : **« combien de cycles puis-je encore voler sans risque ? »**

On calibre une borne basse unilatérale à 95 % : avec probabilité ≥ 95 %, la vraie RUL est **au-dessus** de L. Le score est signé (ŷ − y), car seule la surestimation est dangereuse.

In [ ]:
lb = LowerBoundConformal(alpha=0.05).fit(cal.pred, cal.RUL)
print(f"Borne basse = prédiction − {lb.q_:.1f} cycles")
for name, d in [("validation (coupée)", val_cuts), ("test (dernier cycle)", test_last)]:
    L = lb.predict(d.pred)
    print(f"{name:<22}: vraie RUL ≥ borne dans {np.mean(d.RUL >= L):.1%} des cas (objectif ≥ 95 %)")

t = test_last.assign(L=lb.predict(test_last.pred)).sort_values("RUL").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(t.index, t.L, t.pred, color="seagreen", alpha=0.5, label="de la borne basse à la prédiction")
ax.scatter(t.index, t.RUL, s=12, color="black", zorder=3, label="vraie RUL")
ax.scatter(t.index[t.RUL < t.L], t.RUL[t.RUL < t.L], s=40, color="red", zorder=4, label="borne violée")
ax.set(xlabel="Moteurs de test (triés par vraie RUL)", ylabel="RUL (cycles)", title="Borne basse garantie à 95 % – jeu de test")
ax.legend()
plt.show()

## 11. Sauvegarde

In [ ]:
joblib.dump({"split": split, "mondrian": mond, "lower_bound": lb, "cqr": cqr, "features": FEATS, "alpha": ALPHA},
            MODELS / "conformal_fd001.joblib")
summary.to_csv(PROJECT_ROOT / "reports" / "conformal_summary.csv", index=False)
print("Sauvegardé")

## 12. Conclusions

_À compléter :_
- La couverture globale est-elle tenue sur le test (≥ 90 %) ?
- Quelle méthode garantit le mieux la couverture dans chaque tranche ?
- Quelle méthode donne les intervalles les plus étroits ?
- La borne basse à 95 % est-elle respectée ? De combien de cycles faut-il « reculer » par rapport à la prédiction ?